In [ ]:
from openai import OpenAI

ollama = OpenAI(base_url= 'http://192.168.1.50:11434/v1', api_key='ollama')

response = ollama.chat.completions.create(model='gpt-oss:20b', 
                                          messages=[{'role':'user','content':'tell me a fun fact ?'}],
                                          )
response.choices[0].message.content

"Did you know that **octopuses have three hearts?** \n\nTwo of their hearts pump blood to the gills, while the third pumps it to the rest of the body. And when an octopus swims, the heart that supplies the rest of its body actually stops beating for a moment—so that's why these clever creatures prefer crawling; it’s less exhausting for their circulatory system!"

imports


In [36]:
import os
import json
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI



fetch links from imrankazi.com



In [16]:
links = fetch_website_links('https://imrankazi.com')
links


['https://www.facebook.com/sharer.php?u=https://imrankazi.com/&t=Imran+Kazi+Slideshow',
 'https://twitter.com/home?status=Imran+Kazi+Slideshow+https://imrankazi.com/',
 'http://linkedin.com/shareArticle?mini=true&url=https://imrankazi.com/&title=Imran+Kazi+Slideshow',
 'https://plus.google.com/share?url=https://imrankazi.com/',
 'http://reddit.com/submit?url=https://imrankazi.com/&title=Imran+Kazi+Slideshow',
 'https://www.tumblr.com/share/link?url=https://imrankazi.com/&name=Imran+Kazi+Slideshow&description=Imran+Kazi+Slideshow',
 'https://pinterest.com/pin/create/bookmarklet/?media=&url=https://imrankazi.com/&is_video=false&description=Imran+Kazi+Slideshow',
 '/cdn-cgi/l/email-protection#23464e424a4f63424747514650500d404c4e1c505641494640571e6a4d5746514650574a4d440611136f4a4d48050013101b18414c475a1e6a4e51424d0611136842594a061113704f4a4746504b4c540611134b57575350190c0c4a4e51424d4842594a0d404c4e0c',
 'https://imrankazi.com/',
 'https://www.imrankazi.com',
 'https://imrankazi.com/',
 'ht

Ollama Configuration


In [24]:
import requests
ollama = OpenAI(base_url="http://192.168.1.50:11434/v1", api_key="ollama")
requests.get("http://192.168.1.50:11434").content

b'Ollama is running'

response


In [14]:
def get_links_user_prompts(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt




In [25]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [26]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
       
        model = 'gpt-oss:20b',
        messages=[
            {'role': 'system','content':link_system_prompt},
            {'role': 'user','content':get_links_user_prompts(url)}                  
        ],
        response_format={"type": "json_object"}
        
    )
    result = response.choices[0].message.content 
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")

    return links
    

    
    

In [27]:
select_relevant_links('https://imrankazi.com')


Found 6 relevant links


{'links': [{'type': 'home page', 'url': 'https://imrankazi.com/'},
  {'type': 'about page', 'url': 'https://imrankazi.com/about-me/'},
  {'type': 'contact page', 'url': 'https://imrankazi.com/contact-form/'},
  {'type': 'instagram page', 'url': 'https://imrankazi.com/instagram/'},
  {'type': 'facebook page', 'url': 'https://www.facebook.com/imran.kazi.76'},
  {'type': '500px profile', 'url': 'https://500px.com/imrankazi'}]}

Brochure


In [28]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [29]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""



In [30]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [31]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found 7 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nCloudflare/clef\nUpdated\n5 days ago\n•\n5.42k\n•\n1.55k\nabenzerps/Qwen-Image-2.1-Uncensored-GGUF\nUpdated\n8 days 

In [60]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="gpt-oss:20b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [61]:
create_brochure("HuggingFace", "https://huggingface.co")

Found 15 relevant links


# Hugging Face  
*The AI community building the future.*

---

## 🚀 About Us  
Hugging Face is the world’s largest collaborative hub for machine‑learning.  
- **Model hub:** 2 + million ready‑to‑use models (LLMs, vision, audio, etc.).  
- **Dataset repository:** 500 k + datasets spanning every domain.  
- **Spaces:** 1 + million open‑source AI demos that run on zero‑GPU or custom hardware.  
- **Community:** Thousands of researchers, engineers, and hobbyists publish, fork, and improve code together.  
- **Open‑source first:** Every model and dataset is publicly available with permissive licenses.

---

## 🔧 What We Provide  

| Feature | Highlights | Who Benefits |
|---------|------------|--------------|
| **Inference Services** | End‑to‑end pipelines, GPU/CPU scaling, real‑time APIs | Data scientists, product teams, research labs |
| **Enterprise Plans** | Hugging Face PRO: private model hosts, SLAs, dedicated support | Medium‑to‑large companies, regulated industries |
| **Storage & Compute** | Buckets for data, Zero GPU for free hosting | Startup ML engineers, academic projects |
| **Custom Training** | Fine‑tune any public model quickly with our APIs | AI startups, content creators, R&D teams |
| **Community & Events** | Discord, GitHub, Blogs, Daily Papers, Hands‑on workshops | Anyone eager to learn or share |
| **Hardware & Ops** | Partners with major cloud providers for optimized inference | Enterprises wanting cost‑effective deployment |

---

## 🌍 Why Join the Hugging Face Ecosystem?  

- **Unmatched Scale:** Access to 2 + M models, 500 k + datasets, 1 M+ applications.  
- **Collaboration by Design:** Fork, star, and improve anyone’s work—no barriers to innovation.  
- **Open‑Source Culture:** Contribute code, data, or docs and get credit on a public platform.  
- **Enterprise‑Ready:** Proven infrastructure for high‑availability, compliance, and cost‑control.  
- **Future‑oriented Projects:** Work on state‑of‑the‑art LLMs (e.g., Qwen, JEV), reinforcement learning, video generation, etc.  

---

## 📈 Customers Who Trust Us  

| Company | Use‑case | Result |
|---------|----------|--------|
| **Cloudflare** | Fast, GDPR‑compliant inference for content filtering | 30 % latency drop |
| **Microsoft** | Fine‑tune BERT for enterprise search | 2× relevance improvement |
| **Healthcare Start‑up** | NLP on medical records | 15 % diagnosis accuracy increase |
| **Universities** | Research on multi‑modal datasets | Award‑winning papers |

*(These examples illustrate the breadth of Hugging Face’s impact. We work across tech, finance, health, media, and academia.)*

---

## 👩‍💻 Careers at Hugging Face  

Hugging Face is hiring in the following areas:

- Machine‑Learning Research (NLP, Computer Vision, Reinforcement Learning)  
- Software Engineering (Model Serving, Infrastructure, Front‑end)  
- Data Engineering & Operations (Data Pipelines, Cloud Ops)  
- Community & Education (Content Creation, Outreach, Support)  
- Product, Business Development, and Enterprise Engineering

**Why Work Here?**

- **Open Culture:** Code‑first, data‑first, inclusive.  
- **Impact:** Influence the tools that shape AI for the next generation.  
- **Growth:** Frequent learning streams, hackathons, paid conferences.  
- **Flexibility:** Remote‑first with a distributed team across 30+ countries.

> *“We’re building a commons for AI. Join us and become part of the future.”* – Team Lead, Hugging Face

---

## 🌟 Get Started / Connect  

- **Explore models:** [huggingface.co/models](https://huggingface.co/models)  
- **Clone a Space:** [huggingface.co/spaces](https://huggingface.co/spaces)  
- **Join the community:** Discord, GitHub, Blog, Forums  
- **Enterprise inquiries:** [contact@huggingface.co](mailto:contact@huggingface.co)  

### Contact us  
🏢 **Headquarters** — Paris, France  
✉️ **Email** — sales@huggingface.co | careers@huggingface.co  

---

### Hugging Face — *Because building better AI starts with community.*

In [62]:
create_brochure("Imran Kazi", "https://imrankazi.com")

Found 9 relevant links


# Imran Kazi Photography – Capturing Moments, Telling Stories  

---

## About Us  
*Imran Kazi* is a passionate, self‑taught photographer whose journey began with a Cadbury *Micro 110* camera earned through years of collecting wrappers. Today, he turns everyday scenes into compelling stories, blending technical skill with an emotional connection to his subjects.

- **Vision** – Every photograph is an experience; it tells a unique story in a single frame.  
- **Philosophy** – Good photography is a dialogue between light, subject and story, not just technical perfection.  
- **Approach** – He builds relationships with clients, making them comfortable to reveal their true personalities.

---

## Services & Portfolio  
| Category | Highlights |
|----------|------------|
| **Architectural** | Detailed shots of buildings and interiors, emphasizing structure and light. |
| **Landscape** | Evocative vistas that capture the mood of nature. |
| **Proofings** | High‑resolution editions for clients requiring crisp, print‑ready images. |
| **Events** | Coverage of cultural celebrations, e.g., *Chinese New Year*, and celebrity events. |

> **Featured Projects**  
> • *Nimrahs Closet* – Fashion editorial series  
> • *Simi April 2023* – Portfolio showcase  
> • *Caren 2022 / April 2023* – Behind‑the‑scenes photography  

---

## Notable Clients & Collaborations  
Imran has worked with a range of high‑profile personalities and events, including:

- **John Abraham** – Bollywood film star  
- **Armaan Malik** – Renowned singer  
- **Michael Kinko** – International photographer & collaborator  

This experience underlines his ability to adapt to diverse styles, from celebrity portraits to culturally significant events.

---

## Why Choose Imran Kazi Photography?  

- **Individual Attention** – As a solo artist, every project receives personalized care.  
- **Story‑Driven Imagery** – Each image reflects a narrative that resonates emotionally.  
- **Professional Quality** – Equipped with manual and digital gear, ensuring technical excellence.  

---

## Contact & Booking  

> **Website** – *Imran Kazi’s Personal Photography Site*  
> **Email** – available via the contact form  
> **Instagram** – follow for the latest work and behind‑the‑lens moments  
> **Contact Form** – Use the site’s form to request services or discuss collaborations.  

> *For copyright inquiries or commercial licensing, please use the contact form.*

---

*© 2023 Imran Kazi – All rights reserved*

In [ ]:
def stream_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="gpt-oss:20b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream= True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [35]:
stream_brochure('Imran Kazi', 'https://imrankazi.com')

Found 7 relevant links


NameError: name 'stream' is not defined